# Benchmarks

## Initialize

In [ ]:
#library(Rmisc)
#library(dtplyr)
library(tidyverse)
library(glue)
library(arrow)
library(patchwork)
#library(data.table)
library("jsonlite")
library(ggthemes)

In [ ]:
if (grepl("sc", Sys.info()[["nodename"]], fixed=TRUE)) {
    base_path = "/sc-projects/sc-proj-ukb-cvd"
} else {
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"}
print(base_path)

project_label="22_medical_records"
project_path = glue("{base_path}/results/projects/{project_label}")
figure_path = glue("{project_path}/figures")
output_path = glue("{project_path}/data")

experiment = "220627_covid"
experiment_path = glue("{output_path}/{experiment}")

In [ ]:
library(data.table)
library(arrow)

In [ ]:
endpoints_md = fread(glue("{experiment_path}/endpoints.csv"), colClasses=c("phecode"="character"))
endpoints = sort(endpoints_md$endpoint)

In [ ]:
endpoint_defs = arrow::read_feather(glue("{output_path}/phecode_defs_220306.feather")) %>% arrange(endpoint)

In [ ]:
endpoint_defs %>% group_by(phecode_category) %>% tally()

In [ ]:
endpoint_defs %>% filter(phecode_category %in% c("Death", "ID", "Pulmo", "Cardio")) %>% group_by(phecode_category) %>% tally()

In [ ]:
data_outcomes = arrow::read_feather(glue("{output_path}/baseline_outcomes_220627.feather", as_data_frame=FALSE)) 

In [ ]:
records_long = arrow::read_feather("/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/artifacts/final_records_omop_220531.feather",
                                  col_select=all_of(c("eid", "birth_date", "exit_date", "concept_id", "date", "vocabulary", "origin"))) %>% 
                filter(vocabulary=="phecode") %>% mutate(concept_id = str_replace_all(concept_id, "\\.", "\\-")) %>%
        mutate(severity = case_when(
        origin %in% c("gp_ctv3", "gp_sct") ~ "Light",
        origin %in% c("hes_icd9", "hes_icd10") ~ "Severe",
        origin == "death_records" ~ "Fatal")) %>%
    mutate(severity = factor(severity, levels=c("Light", "Severe", "Fatal")))

In [ ]:
partitions = 0:21
partitions
paths = c()
for (p in partitions){
    temp_path = glue("{experiment_path}/loghs/Identity(Records)+MLP/{p}/test.feather")
    paths = c(paths, temp_path)
    }

In [ ]:
predictions = paths %>% map_df(
    ~suppressWarnings(read_feather(.))) %>% 
    #pivot_longer(endpoint_selection, names_to="endpoint", values_to="logh") %>%
    mutate(eid = as.integer(as.character(eid))) #%>%
    #select(endpoint, eid, logh))# %>% arrange(endpoint, eid)
predictions_nocovid = predictions %>% arrange(eid)
predictions_nocovid %>% head()

In [ ]:
data_covariates = arrow::read_feather(glue("{output_path}/220627/data_covariates_full.feather"))

In [ ]:
#predictions = bind_rows(predictions_recruitment %>% mutate(t0="covid", endpoints="), predictions_2020 %>% mutate(t0="covid")) %>% left_join(data_covariates)

In [ ]:
predictions = predictions_nocovid %>% left_join(data_covariates)

In [ ]:
endpoints_md %>% filter(str_detect(phecode_string, "Death|Pneumonia$|Sepsis$"))

In [ ]:
endpoints_md %>% 
    #filter(str_detect(phecode_string, "virus")) %>% 
    filter(str_detect(phecode_category, "ID"))

In [ ]:
endpoints_md %>% 
    filter(str_detect(phecode_string, "Infections")) #%>% 
    #filter(str_detect(phecode_category, "ID"))

In [ ]:
t0_date = as.Date("2019-12-31")

In [ ]:
library(lubridate)
data_outcomes_long = records_long %>% filter(date>t0_date) %>% filter(t0_date<exit_date) %>% 
    mutate(event=1, time=time_length(difftime(date, t0_date), "years")) %>%
    arrange(eid, concept_id, severity, time) 

In [ ]:
library(survcomp)

In [ ]:
library(survival)
library(survminer)
require("ggquickeda")

mh_map = c("20"="#023768", "10"="#4F8EC1", "1"="#7AC6FF")

calc_km = function(predictors, endpoint, t0, severity){
    temp_outcomes = data_outcomes_long %>% 
        filter(concept_id==!!endpoint) %>%
        filter(severity==!!severity) %>%
        select(eid, event, time) %>% group_by(eid) %>% slice_min(time, with_ties=FALSE) %>% ungroup()
    
    km_df = predictions %>% 
        filter(t0==!!t0) %>%
        select(eid, all_of(predictors)) %>% 
        mutate(across(all_of(predictors), ~ntile(., 100))) %>%
        mutate(predictor = rowSums(across(all_of(predictors)))) %>%
        left_join(temp_outcomes, by="eid") %>%
        replace_na(list(event=FALSE, time=time_length(difftime(max(data_outcomes_long$exit_date), t0_date), "years"))) %>%
        rename(logh=predictor) %>% ungroup() %>%
        mutate(logh_bin = factor(ntile(logh, 20)))
    return(km_df)
    }

get_cindex_df = function(predictors, endpoint, t0, severity){
    temp_outcomes = data_outcomes_long %>% 
        filter(concept_id==!!endpoint) %>%
        filter(severity==!!severity) %>%
        select(eid, event, time) %>% group_by(eid) %>% slice_min(time, with_ties=FALSE) %>% ungroup()
    
    km_df = predictions %>% 
        filter(t0==!!t0) %>%
        select(eid, all_of(predictors)) %>% 
        mutate(across(all_of(predictors), ~ntile(., 100))) %>%
        mutate(predictor = rowSums(across(all_of(predictors)))) %>%
        left_join(temp_outcomes, by="eid") %>%
        replace_na(list(event=FALSE, time=time_length(difftime(max(data_outcomes_long$exit_date), t0_date), "years"))) %>%
        rename(logh=predictor) %>%
        mutate(logh_bin = factor(ntile(logh, 20)))
    
    return(km_df %>% mutate(predictors = paste(predictors, collapse=", "), severity=severity))
    }

plot_km = function(predictors, endpoint, t0, severity){
    temp_outcomes = data_outcomes_long %>% 
        filter(concept_id==!!endpoint) %>%
        filter(severity==!!severity) %>%
        select(eid, event, time) %>% group_by(eid) %>% slice_min(time, with_ties=FALSE) %>% ungroup()
    
    km_df = predictions %>% 
        filter(t0==!!t0) %>%
        select(eid, all_of(predictors)) %>% 
        mutate(across(all_of(predictors), ~ntile(., 100))) %>%
        mutate(predictor = rowSums(across(all_of(predictors)))) %>%
        left_join(temp_outcomes, by="eid") %>%
        replace_na(list(event=FALSE, time=time_length(difftime(max(data_outcomes_long$exit_date), t0_date), "years"))) %>%
        rename(logh=predictor) %>%
        mutate(logh_bin = factor(ntile(logh, 20)))
    
    
    
   #print(km_df)
    cindex_df = km_df %>% sample_n(100000)
    print(severity)
    print((concordance.index(x=cindex_df$logh, surv.time=cindex_df$time, surv.event = cindex_df$event))$c.index)
    
    ratio = round((km_df %>% summarise(ratio = mean(event)))$ratio*100, 2)
    #print(ratio)
    
    
    g = ggplot(km_df %>% filter(logh_bin %in% c(1, 10, 20)), aes(time = time, status = event, fill=logh_bin, color=logh_bin, group=logh_bin)) +
    
        labs(title=glue("{severity} ({ratio}%)"), y="Cumulative Incidence (%)", x=NULL) + 
        geom_hline(yintercept=as.numeric(ratio)/100, color="black", alpha=0.5, linetype="21")+
    
        annotate("rect", xmin=0.22, xmax=0.45, ymin=0, ymax=Inf, fill="black", alpha=0.02) +
        annotate("rect", xmin=0.7, xmax=1.25, ymin=0, ymax=Inf, fill="black", alpha=0.02) +
    
        annotate("text", label="First\nWave", x=0.335, y=0.07, hjust=0.5)+#, size=3) +
        annotate("text", label="Second\nWave", x=0.99, y=0.07, hjust=0.5)+#, size=3) +
    
        geom_km(trans = "event") + 
        geom_kmticks(trans = "event", size=0.2, alpha=0.01) + 
        geom_kmband(trans = "event") + 
    
        scale_color_manual(values=mh_map)+scale_fill_manual(values=mh_map)+
    
        scale_y_continuous(labels = function(x) round(x*100, 1), expand=c(0, 0))+
        scale_x_continuous(expand=expansion(add=c(0, .1)), 
                           breaks=c(0, 0.25, 0.5, 0.75, 1, 1.25, 1.5), 
                           labels=c("Jan 20", "April 20", "July 20", "Oct 20", "Jan 21", "April 21", "July 21")) +
        coord_cartesian(xlim=c(0, 1.6), ylim=c(0,0.025))+#+
                           
        theme(legend.position="none", plot.title = element_text(hjust = 0.5))                    
    
    return(g)
    }

In [ ]:
breaks=c(0, 0.25, 0.5, 0.75, 1, 1.25, 1.5)
labels=c("Jan 20", "April 20", "July 20", "Oct 20", "Jan 21", "April 21", "July 21")

#end of first wave 0.45
#end of second wave 0.99
#August 2021 1.6

In [ ]:
base_size = 8
title_size = 10
facet_size = 8.5
geom_text_size=3
theme_set(theme_classic(base_size = base_size) + 
          theme(strip.background = element_blank(), plot.title=element_text(size=title_size, hjust=0), 
                strip.text.x = element_text(size = facet_size),axis.title=element_text(size=10), axis.text=element_text(size=8, color="black"),
                legend.position="bottom", axis.line = element_line(size = 0.2), axis.ticks=element_line(size=0.2)))

In [ ]:
plot_width = 4.125; plot_height=6; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)

In [ ]:
endpoint = "phecode_059"

predictors= c("age")
cindex_df_age = bind_rows(get_cindex_df(predictors, endpoint, "covid", "Severe"), get_cindex_df(predictors, endpoint, "covid", "Fatal"))

predictors= c("age", "OMOP_4306655", "phecode_468", "phecode_092-2")
cindex_df_agemh = bind_rows(get_cindex_df(predictors, endpoint, "covid", "Severe"), get_cindex_df(predictors, endpoint, "covid", "Fatal"))

cindex_df = bind_rows(cindex_df_age, cindex_df_agemh)

In [ ]:
cindex_df %>% write_feather("outputs/covid_cindeces_221020.feather")

In [ ]:
predictors= c("age")
endpoint = "phecode_059"
predictor_label = toString((endpoints_md %>% filter(endpoint %in% predictors))$phecode_string)

g1 = plot_km(predictors, endpoint, "covid", "Severe")
df_1 = calc_km(predictors, endpoint, "covid", "Severe") %>% mutate(score = "age only")
g2 = plot_km(predictors, endpoint, "covid", "Fatal")
df_2 = calc_km(predictors, endpoint, "covid", "Fatal") %>% mutate(score = "age only")

strat_age = (g1/g2) + plot_annotation(title = glue('COVID outcomes stratified by {predictors}'))
strat_age 

In [ ]:
predictors= c("age", "OMOP_4306655", "phecode_468", "phecode_092-2")
endpoint = "phecode_059"
predictor_label = toString((endpoints_md %>% filter(endpoint %in% predictors))$phecode_string)

g3 = plot_km(predictors, endpoint, "covid", "Severe")
df_3 = calc_km(predictors, endpoint, "covid", "Severe") %>% mutate(score = "age+mh")
g4 = plot_km(predictors, endpoint, "covid", "Fatal")
df_4 = calc_km(predictors, endpoint, "covid", "Fatal") %>% mutate(score = "age+mh")

strat_score = (g3/g4) + plot_annotation(title = glue('COVID outcomes stratified by aggregated\npartial hazards of {predictor_label}'))
strat_score

In [ ]:
plot_width = 8.25; plot_height=6; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)

strat_age | strat_score

In [ ]:
library(broom)

In [ ]:
colnames(predictions)

In [ ]:
age_new = arrow::read_feather('/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/220603_medicalhistory/baseline_covariates.feather', 
                                       col_select=c("eid", "age_at_recruitment_f21022_0_0", "date_of_attending_assessment_centre_f53_0_0")) %>%
    mutate(date_covid = as.Date("2019-12-31")) %>%
    mutate(diff_covid = time_length(difftime(date_covid, date_of_attending_assessment_centre_f53_0_0), "years")) %>%
    mutate(age_covid = floor(age_at_recruitment_f21022_0_0 + diff_covid)) %>% select(eid, age_covid)

In [ ]:
predictions %>% left_join(age_new) %>% select(eid, age, age_covid) %>% mutate(age_bin=ntile(age, 20)) %>% filter(age_bin==20) %>% arrange(age_covid) %>% ggdist::median_qi(age_covid)

In [ ]:
calc_km_df = function(df){
    fit <- survfit(Surv(time, event) ~ logh_bin, data=df %>% filter(logh_bin==20))
    surv_summary(fit) %>% 
        mutate(cumevents = cumsum(n.event)) %>% 
        mutate(label = glue("{round((1-surv)*100, 2)}% (CI {round((1-upper)*100, 2)}%, {round((1-lower)*100, 2)}%), {cumevents} events")) %>%
        mutate(link="yes") %>% left_join(data.frame(link = c("yes", "yes", "yes"), times = c(0.45, 0.99, 1.6)), by="link") %>%
        filter(time<times) %>% group_by(times) %>% slice_max(time) %>% ungroup()
    }

In [ ]:
km_df_all = bind_rows(
    calc_km_df(df_1) %>% mutate(score="age only", severity="Severe"),
    calc_km_df(df_2) %>% mutate(score="age+only", severity="Fatal"),
    calc_km_df(df_3) %>% mutate(score="age+hr", severity="Severe"),
    calc_km_df(df_4) %>% mutate(score="age+hr", severity="Fatal")
    ) %>%
    select(times, score, severity, label) %>% pivot_wider(names_from="times", values_from="label") %>% arrange(desc(severity))

In [ ]:
colnames(km_df_all) = c("score", "severity", "after first wave", "after second wave", "until August 21")

In [ ]:
km_df_all

In [ ]:
summary(km_fit, times = c(0.45, 0.99, 1.6))$table

In [ ]:
str(summary(km_fit, times = c(0.45, 0.99, 1.6)))

In [ ]:
p = g1
layer_data(p, 7) %>% filter(x<=0.45) %>% group_by(colour) %>% slice_max(time, n=2) %>% filter(y<1) %>% slice_max(time, n=1) %>% ungroup()

In [ ]:
calc_survival = function(p, time){
    line_df = layer_data(p, 7) %>% filter(x<=!!time) %>% group_by(colour) %>% slice_max(time, n=2) %>% filter(y<1) %>% slice_max(time, n=1) %>% ungroup()
    fill_df = layer_data(p, 8) %>% filter(x<=!!time) %>% group_by(colour) %>% slice_max(time, n=1) %>% slice_max(upper, n=1, with_ties=FALSE) %>% ungroup()# %>% filter(y<1)
    survival_df = line_df %>% select(group, x, time, survival) %>% left_join(fill_df %>% select(group, lower, upper), by=c("group")) %>% select(group, x, time, lower, survival, upper)
    survival_df_clean = survival_df %>% mutate(label = glue("{round(survival*100, 2)}% (CI {round(upper*100, 2)}%, {round(lower*100, 2)}%)"))
    return(survival_df_clean)
    }

In [ ]:
calc_survival(g1, 0.45)

In [ ]:
calc_survival(g2, 0.45)

In [ ]:
calc_survival(g1, 0.99)

In [ ]:
calc_survival(g2, 0.99)

In [ ]:
predictors= c("age", 
              'phecode_488-9',
              'phecode_719-8',
              'phecode_333-1',
              'phecode_164',
              'phecode_501',
              'phecode_247',
              'phecode_061',
              'phecode_232-29',
              'phecode_526-2',
              'phecode_542',
              'phecode_114-6',
              'phecode_164-1',
              'phecode_160-1',
              'phecode_247-42',
              'phecode_160',
              'phecode_240',
              'phecode_335-4',
              'phecode_089-1',
              'phecode_252',
              'phecode_325-2',
              'phecode_707-4',
              'phecode_325',
              'phecode_247-7',
              'phecode_468-14',
              'phecode_247-72',
              'phecode_417-2',
              'phecode_823-1',
              'phecode_337-8',
              'phecode_468-1',
              'phecode_510-7',
              'phecode_726-5',
              'phecode_477-2',
              'phecode_686-2',
              'phecode_164-2',
              'phecode_280-2',
              'phecode_546-3',
              'phecode_002-1',
              'phecode_256-1',
              'phecode_160-2',
              'phecode_232-2',
              'phecode_498',
              'phecode_343-1',
              'phecode_805',
              'phecode_513-2',
              'phecode_973',
              'phecode_545',
              'phecode_557-1',
              'phecode_337',
              'phecode_550-1',
              'phecode_098'
             )
endpoint = "phecode_059"
predictor_label = toString((endpoints_md %>% filter(endpoint %in% predictors))$phecode_string)

g3 = plot_km(predictors, endpoint, "covid", "Severe")
df_3 = calc_km(predictors, endpoint, "covid", "Severe") %>% mutate(score = "age+mh")
g4 = plot_km(predictors, endpoint, "covid", "Fatal")
df_4 = calc_km(predictors, endpoint, "covid", "Fatal") %>% mutate(score = "age+mh")

strat_score = (g3/g4) + plot_annotation(title = glue('COVID outcomes stratified by aggregated\npartial hazards of {predictor_label}'))
strat_score

In [ ]:
plot_width = 4.125; plot_height=4; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
ggplot(bind_rows(df_1, df_3) %>% group_by(score, logh_bin) %>% summarise(ratio=sum(event)/n()), aes(x=logh_bin, y=ratio, color=score)) + geom_point()

In [ ]:
plot_width = 4.125; plot_height=4; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
ggplot(bind_rows(df_2, df_4) %>% group_by(score, logh_bin) %>% summarise(ratio=sum(event)/n()), aes(x=logh_bin, y=ratio, color=score)) + geom_point()

In [ ]:
ggplot(df_3 %>% group_by(logh_bin) %>% summarise(ratio=sum(event)/n()), aes(x=logh_bin, y=ratio)) + geom_point()

In [ ]:
ggplot(df_2 %>% group_by(logh_bin) %>% summarise(ratio=sum(event)/n()), aes(x=logh_bin, y=ratio)) + geom_point()

In [ ]:
ggplot(df_4 %>% group_by(logh_bin) %>% summarise(ratio=sum(event)/n()), aes(x=logh_bin, y=ratio)) + geom_point()

In [ ]:
ggplot(df_1 %>% group_by(logh_bin) %>% summarise(ratio=sum(event)), aes(x=logh_bin, y=ratio)) + geom_point()

In [ ]:
ggplot(df_1, aes(as.numeric(logh_bin), color=factor(event), fill=factor(event))) + geom_density(alpha=0.1)

In [ ]:
ggplot(bind_rows(df_1, df_3) %>% filter(event==1), aes(as.numeric(logh_bin), color=factor(score), fill=factor(score))) + geom_density(alpha=0.1)

In [ ]:
ggplot(bind_rows(df_2, df_4) %>% filter(event==1), aes(as.numeric(logh_bin), color=factor(score), fill=factor(score))) + geom_density(alpha=0.1)

In [ ]:
ggplot(df_3, aes(as.numeric(logh_bin), color=factor(event), fill=factor(event))) + geom_density(alpha=0.1)

In [ ]:
ggplot(df_2, aes(as.numeric(logh_bin), color=factor(event), fill=factor(event))) + geom_density(alpha=0.1)

In [ ]:
ggplot(df_4, aes(as.numeric(logh_bin), color=factor(event), fill=factor(event))) + geom_density(alpha=0.1)

In [ ]:
calc_survival(g3, 0.45)

In [ ]:
calc_survival(g4, 0.45)

In [ ]:
calc_survival(g3, 0.99)

In [ ]:
calc_survival(g4, 0.99)

In [ ]:
predictions

In [ ]:
predictors= c("OMOP_4306655", "phecode_468",  "phecode_092-2")
endpoint = "phecode_059"
predictor_label = toString((endpoints_md %>% filter(endpoint %in% predictors))$phecode_string)

g2 = plot_km(predictors, endpoint, "covid", "Severe")
g3 = plot_km(predictors, endpoint, "covid", "Fatal")

(g2 | g3) + plot_annotation(title = glue('COVID outcomes stratified by {predictor_label} partial hazard'))

In [ ]:
predictors= c("OMOP_4306655", "phecode_468",  "phecode_092-2")
endpoint = "phecode_059"
predictor_label = toString((endpoints_md %>% filter(endpoint %in% predictors))$phecode_string)

g2 = plot_km(predictors, endpoint, "covid", "Severe")
g3 = plot_km(predictors, endpoint, "covid", "Fatal")

(g2 | g3) + plot_annotation(title = glue('COVID outcomes stratified by {predictor_label} partial hazard'))

In [ ]:
predictor = "phecode_061"	
endpoint = "phecode_059"
predictor_label = (endpoints_md %>% filter(endpoint==!!predictor))$phecode_string

g2 = plot_km("covid", "Severe")
g3 = plot_km("covid", "Fatal")

(g2 | g3) + plot_annotation(title = glue('COVID outcomes stratified by {predictor_label} partial hazard'))

In [ ]:
phecode_061

In [ ]:
phecode_089

In [ ]:
predictor = "phecode_092"	
endpoint = "phecode_059"
predictor_label = (endpoints_md %>% filter(endpoint==!!predictor))$phecode_string

g1 = plot_km("covid", "Light")
g2 = plot_km("covid", "Severe")
g3 = plot_km("covid", "Fatal")

(g1 | g2 | g3) + plot_annotation(title = glue('COVID outcomes stratified by {predictor_label} partial hazard'))

In [ ]:
length(unique(predictions$eid))

In [ ]:
predictions

In [ ]:
base_size = 8
title_size = 10
facet_size = 8.5
geom_text_size=3
theme_set(theme_classic(base_size = base_size) + 
          theme(strip.background = element_blank(), plot.title=element_text(size=title_size, hjust=0), 
                strip.text.x = element_text(size = facet_size),axis.title=element_text(size=10), axis.text=element_text(size=8, color="black"),
                legend.position="bottom", axis.line = element_line(size = 0.2), axis.ticks=element_line(size=0.2)))

# Figure 2: Selected Endpoints

In [ ]:
pred_outcomes = predictions %>% left_join(data_outcomes, on=c(eid, endpoint)) %>% as_tibble()

## Endpoint Prevalence + Rate Ratios

In [ ]:
logh_inc = pred_outcomes %>% filter(included==1) %>% group_by(endpoint) %>% mutate(logh_perc = ntile(logh, 10)) %>% ungroup() %>% as_tibble()

In [ ]:
logh_T_agg = logh_inc %>% group_by(endpoint, logh_perc) %>% summarise(n=sum(event), ratio = mean(event)) %>% as_tibble()

In [ ]:
temp = logh_T_agg %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% 
    filter(endpoint %in% endpoint_selection) %>% ungroup() #%>% sample_n(10000)
temp %>% filter(logh_perc %in% c(1, 10)) %>% 
    pivot_wider(names_from=logh_perc, values_from=c(n, ratio)) %>% 
    mutate(ratio=ratio_10/ratio_1) %>%
    mutate(endpoint = recode(endpoint, !!!endpoint_map)) %>%
    #mutate(ratio_1=ratio_1*100, ratio_10 = ratio_10*100) %>% 
    arrange(ratio)

## Endpoint selection for health state and incident disease rates

In [ ]:
pred_outcomes = predictions %>% left_join(data_outcomes, on=c(eid, endpoint)) %>% as.data.table() 

## Example illustration with deciles

In [ ]:
logh_inc = pred_outcomes %>% filter(included==1) %>% group_by(endpoint) %>% mutate(logh_perc = ntile(logh, 10)) %>% ungroup() %>% as_tibble()

In [ ]:
logh_T_agg = logh_inc %>% group_by(endpoint, logh_perc) %>% summarise(n=sum(event), ratio = mean(event)) %>% as_tibble()

In [ ]:
#logh_T_endpoint = logh_inc %>% group_by(endpoint) %>% summarise(n_all=sum(event), ratio_all = mean(event)) %>% as_tibble()

In [ ]:
endpoint_order

In [ ]:
plot_width = 2; plot_height=2; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
#temp_rank = event_rest %>% filter(features=="Metabolomics") %>% arrange(desc(MET10PercvsREST))
#endpoint_order = (endpoint_defs %>% mutate(phecode_rank = as.numeric(phecode)) %>% arrange(phecode_rank) %>% as_tibble())$endpoint
temp = logh_T_agg %>% filter(endpoint=="OMOP_4306655") %>% ungroup() %>% arrange(endpoint) #%>% sample_n(10000)

fig2a = ggplot(temp, aes(x=as.numeric(as.character(logh_perc)), y=ratio*100, color=logh_perc)) + 
    labs(title="All-cause Death", x="Risk Percentile (%)", y="Incident Events (%)") +
    geom_line(alpha=0.7, size=0.3) +    
    geom_point(alpha=0.7, size=0.3) + 
    geom_point(data=temp %>% filter(logh_perc %in% c(1, 10)), alpha=1, size=1, color="black") + 
    geom_hline(aes(yintercept=0.361170266*100), alpha=0.3, linetype="22", size=0.25) + 
    geom_hline(aes(yintercept=0.003323648*100), alpha=0.3, linetype="22", size=0.25) + 
    geom_text(data=temp %>% filter(logh_perc==1), mapping=aes(label=glue("Bottom 10%: {n} ({round(ratio*100, 1)}%)")), color="black", size=2, x=1, y=27, hjust=0) + 
    geom_text(data=temp %>% filter(logh_perc==10), mapping=aes(label=glue("Top 10%: {n} ({round(ratio*100, 1)}%)")), color="black", size=2, x=1, y=30, hjust=0) + 
    geom_text(data=temp %>% pivot_wider(names_from=logh_perc, values_from=c("n", "ratio")), 
              mapping=aes(label=glue("Rate Ratio: ({round(ratio_10*100, 1)}% / {round(ratio_1*100, 1)}%)")), 
              color="black", size=2, x=1, y=24, hjust=0, alpha=0.9) + 
    geom_text(data=temp %>% pivot_wider(names_from=logh_perc, values_from=c("n", "ratio")), 
              mapping=aes(label=glue("Rate Ratio: ~ {round(ratio_10/ratio_1, 1)}")), 
              color="black", size=3, x=1, y=20, hjust=0, alpha=0.9) +
    scale_colour_gradient(low = "#7AC6FF", high = "#023768", space = "Lab", na.value = "grey50", guide = "colourbar", aesthetics = "colour")+
    scale_y_continuous(limits=c(0, NA), expand=expansion(mult=c(.02, .02)))+#, limits=c(0, NA))+
    scale_x_continuous(expand=expansion(add=c(0.25, 0.25)), breaks = c(1, 2, 3, 4, 5, 6, 7, 8, 9, 10))+
    theme(legend.position="none", plot.title = element_text(hjust = 0.5))
fig2a

In [ ]:
0.361170266/0.003323648

## Real Figure with percentiles

In [ ]:
logh_inc = pred_outcomes %>% filter(included==1) %>% group_by(endpoint) %>% mutate(logh_perc = ntile(logh, 100)) %>% ungroup() %>% as_tibble()

In [ ]:
logh_T_agg = logh_inc %>% group_by(endpoint, logh_perc) %>% summarise(n=sum(event), ratio = mean(event)) %>% as_tibble()

In [ ]:
#logh_T_endpoint = logh_inc %>% group_by(endpoint) %>% summarise(n_all=sum(event), ratio_all = mean(event)) %>% as_tibble()

In [ ]:
endpoint_order

In [ ]:
plot_width = 8.25; plot_height=6; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
#temp_rank = event_rest %>% filter(features=="Metabolomics") %>% arrange(desc(MET10PercvsREST))
#endpoint_order = (endpoint_defs %>% mutate(phecode_rank = as.numeric(phecode)) %>% arrange(phecode_rank) %>% as_tibble())$endpoint
temp = logh_T_agg %>% filter(endpoint %in% endpoint_selection) %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% ungroup() %>% arrange(endpoint) #%>% sample_n(10000)

fig2a = ggplot(temp, aes(x=logh_perc, y=ratio*100, color=logh_perc)) + 
    labs(title=NULL, x="Risk Percentile (%)", y="Incident Events (%)") +
    geom_point(alpha=0.7, size=0.3) + 
    scale_colour_gradient(low = "#7AC6FF", high = "#023768", space = "Lab", na.value = "grey50", guide = "colourbar", aesthetics = "colour")+
    scale_y_continuous(limits=c(0, NA), expand=expansion(mult=c(0, .05)))+#, limits=c(0, NA))+
    scale_x_continuous(expand=expansion(add=c(0, 1)))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = as_labeller(endpoint_map, default=label_wrap_gen(22))), ncol=6) + theme(legend.position="none")
fig2a

In [ ]:
plot_width = 8.25; plot_height=3.5; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
#temp_rank = event_rest %>% filter(features=="Metabolomics") %>% arrange(desc(MET10PercvsREST))
#endpoint_order = (endpoint_defs %>% mutate(phecode_rank = as.numeric(phecode)) %>% arrange(phecode_rank) %>% as_tibble())$endpoint
temp = logh_T_agg %>% filter(endpoint %in% endpoint_selection) %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% ungroup() %>% arrange(endpoint) #%>% sample_n(10000)

fig2a_common = ggplot(temp %>% filter(endpoint %in% endpoints_common), aes(x=logh_perc, y=ratio*100, color=logh_perc)) + 
    labs(title=NULL, x="Risk Percentile (%)", y="Incident Events (%)") +
    geom_point(alpha=0.7, size=0.3) + 
    scale_colour_gradient(low = "#7AC6FF", high = "#023768", space = "Lab", na.value = "grey50", guide = "colourbar", aesthetics = "colour")+
    scale_y_continuous(limits=c(0, NA), expand=expansion(mult=c(0, .05)))+#, limits=c(0, NA))+
    scale_x_continuous(expand=expansion(add=c(0, 1)))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = as_labeller(endpoint_map, default=label_wrap_gen(22))), ncol=6) + theme(legend.position="none")
fig2a_common

In [ ]:
plot_width = 8.25; plot_height=1.75; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
#temp_rank = event_rest %>% filter(features=="Metabolomics") %>% arrange(desc(MET10PercvsREST))
#endpoint_order = (endpoint_defs %>% mutate(phecode_rank = as.numeric(phecode)) %>% arrange(phecode_rank) %>% as_tibble())$endpoint
temp = logh_T_agg %>% filter(endpoint %in% endpoint_selection) %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% ungroup() %>% arrange(endpoint) #%>% sample_n(10000)

fig2a_cardio = ggplot(temp %>% filter(endpoint %in% endpoints_cardio), aes(x=logh_perc, y=ratio*100, color=logh_perc)) + 
    labs(title=NULL, x="Risk Percentile (%)", y="Incident Events (%)") +
    geom_point(alpha=0.7, size=0.3) + 
    scale_colour_gradient(low = "#7AC6FF", high = "#023768", space = "Lab", na.value = "grey50", guide = "colourbar", aesthetics = "colour")+
    scale_y_continuous(limits=c(0, NA), expand=expansion(mult=c(0, .05)))+#, limits=c(0, NA))+
    scale_x_continuous(expand=expansion(add=c(0, 1)))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = as_labeller(endpoint_map, default=label_wrap_gen(22))), ncol=6) + theme(legend.position="none")
fig2a_cardio

In [ ]:
plot_width = 8.25; plot_height=1.75; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
#temp_rank = event_rest %>% filter(features=="Metabolomics") %>% arrange(desc(MET10PercvsREST))
#endpoint_order = (endpoint_defs %>% mutate(phecode_rank = as.numeric(phecode)) %>% arrange(phecode_rank) %>% as_tibble())$endpoint
temp = logh_T_agg %>% filter(endpoint %in% endpoint_selection) %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% ungroup() %>% arrange(endpoint) #%>% sample_n(10000)

fig2a_interesting = ggplot(temp %>% filter(endpoint %in% endpoints_interesting), aes(x=logh_perc, y=ratio*100, color=logh_perc)) + 
    labs(title=NULL, x="Risk Percentile (%)", y="Incident Events (%)") +
    geom_point(alpha=0.7, size=0.3) + 
    scale_colour_gradient(low = "#7AC6FF", high = "#023768", space = "Lab", na.value = "grey50", guide = "colourbar", aesthetics = "colour")+
    scale_y_continuous(limits=c(0, NA), expand=expansion(mult=c(0, .05)))+#, limits=c(0, NA))+
    scale_x_continuous(expand=expansion(add=c(0, 1)))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = as_labeller(endpoint_map, default=label_wrap_gen(22))), ncol=6) + theme(legend.position="none")
fig2a_interesting

# Kaplan Mayer

In [ ]:
logh_mh = logh_inc %>% select(endpoint, eid, logh_perc, event, time) %>% group_by(endpoint) %>% 
    mutate(MH=case_when(logh_perc %in% 100 ~ "High", 
                        logh_perc %in% 51 ~ "Mid", 
                        logh_perc %in% 1 ~ "Low",
                        TRUE ~ "NA")
          ) %>% mutate(MH=fct_rev(factor(MH, levels=c("Low", "Mid", "High")))) %>% ungroup() %>% 
    filter(MH!="NA") #%>% select(eid, endpoint, logh, logh_group)

In [ ]:
require("ggquickeda")
plot_width = 8.25; plot_height=6; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
met_map = c("High"="#023768", "Mid"="#4F8EC1", "Low"="#7AC6FF")

temp = logh_mh %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% filter(endpoint %in% endpoint_selection) %>% ungroup() 

fig2b = ggplot(temp, aes(time = time, status = event, fill=MH, color=MH, group=MH)) +
    geom_km(trans = "event") + 
    geom_kmticks(trans = "event", size=0.2, alpha=0.01) + 
    geom_kmband(trans = "event") + 
    labs(x="Time (Years)", y="Cumulative Events (%)")+
    scale_color_manual(values=met_map)+scale_fill_manual(values=met_map)+
    scale_y_continuous(labels = function(x) round(x*100, 1), expand=c(0, 0))+
    scale_x_continuous(expand=expansion(add=c(0, .1)), breaks=c(5, 10))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = as_labeller(endpoint_map, default=label_wrap_gen(22))), ncol=6) + theme(legend.position="none")
fig2b

In [ ]:
require("ggquickeda")
plot_width = 8.25; plot_height=3.5; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
met_map = c("High"="#023768", "Mid"="#4F8EC1", "Low"="#7AC6FF")

temp = logh_mh %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% filter(endpoint %in% endpoint_selection) %>% ungroup() 

fig2b_common = ggplot(temp %>% filter(endpoint %in% endpoints_common), aes(time = time, status = event, fill=MH, color=MH, group=MH)) +
    geom_km(trans = "event") + 
    geom_kmticks(trans = "event", size=0.2, alpha=0.01) + 
    geom_kmband(trans = "event") + 
    labs(x="Time (Years)", y="Cumulative Events (%)")+
    scale_color_manual(values=met_map)+scale_fill_manual(values=met_map)+
    scale_y_continuous(labels = function(x) round(x*100, 1), expand=c(0, 0))+
    scale_x_continuous(expand=expansion(add=c(0, .1)), breaks=c(5, 10))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = as_labeller(endpoint_map, default=label_wrap_gen(22))), ncol=6) + theme(legend.position="none")
fig2b_common

In [ ]:
require("ggquickeda")
plot_width = 8.25; plot_height=1.75; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
met_map = c("High"="#023768", "Mid"="#4F8EC1", "Low"="#7AC6FF")

temp = logh_mh %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% filter(endpoint %in% endpoint_selection) %>% ungroup() 

fig2b_cardio = ggplot(temp %>% filter(endpoint %in% endpoints_cardio), aes(time = time, status = event, fill=MH, color=MH, group=MH)) +
    geom_km(trans = "event") + 
    geom_kmticks(trans = "event", size=0.2, alpha=0.01) + 
    geom_kmband(trans = "event") + 
    labs(x="Time (Years)", y="Cumulative Events (%)")+
    scale_color_manual(values=met_map)+scale_fill_manual(values=met_map)+
    scale_y_continuous(labels = function(x) round(x*100, 1), expand=c(0, 0))+
    scale_x_continuous(expand=expansion(add=c(0, .1)), breaks=c(5, 10))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = as_labeller(endpoint_map, default=label_wrap_gen(22))), ncol=6) + theme(legend.position="none")
fig2b_cardio

In [ ]:
require("ggquickeda")
plot_width = 8.25; plot_height=1.75; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
met_map = c("High"="#023768", "Mid"="#4F8EC1", "Low"="#7AC6FF")

temp = logh_mh %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% filter(endpoint %in% endpoint_selection) %>% ungroup() 

fig2b_interesting = ggplot(temp %>% filter(endpoint %in% endpoints_interesting), aes(time = time, status = event, fill=MH, color=MH, group=MH)) +
    geom_km(trans = "event") + 
    geom_kmticks(trans = "event", size=0.2, alpha=0.01) + 
    geom_kmband(trans = "event") + 
    labs(x="Time (Years)", y="Cumulative Events (%)")+
    scale_color_manual(values=met_map)+scale_fill_manual(values=met_map)+
    scale_y_continuous(labels = function(x) round(x*100, 1), expand=c(0, 0))+
    scale_x_continuous(expand=expansion(add=c(0, .1)), breaks=c(5, 10))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = as_labeller(endpoint_map, default=label_wrap_gen(22))), ncol=6) + theme(legend.position="none")
fig2b_interesting

In [ ]:
plot_width = 8.25; plot_height=12; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)

fig2 = (fig2a / fig2b)
fig2

In [ ]:
plot_width = 8.25; plot_height=6; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)

fig2_common = (fig2a_common / fig2b_common)
fig2_common

In [ ]:
plot_width = 8.25; plot_height=3.5; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)

fig2_cardio = (fig2a_cardio / fig2b_cardio)
fig2_cardio

In [ ]:
plot_name = "Figure_2_EventRates"
fig2 %>% ggsave(filename=glue("outputs/{plot_name}.pdf"), device="pdf", width=plot_width, height=plot_height, dpi=plot_res, limitsize=FALSE)

In [ ]:
plot_width = 50; plot_height=75; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
#temp_rank = event_rest %>% filter(features=="Metabolomics") %>% arrange(desc(MET10PercvsREST))
endpoint_order = (phecode_defs %>% mutate(phecode_rank = as.numeric(phecode)) %>% arrange(phecode_rank) %>% as_tibble())$endpoint
endpoint_selection = endpoint_order# %>% head(500)
temp = logh_T_agg %>% filter(model=="GNN(Records)+MLP") %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% filter(endpoint %in% endpoint_selection) %>% ungroup() #%>% sample_n(10000)

mh_events = ggplot(temp, aes(x=logh_perc, y=ratio*100, color=logh_perc)) + 
    labs(title=NULL, x="Medical History Percentile [%]", y="Observed Event Rate [%]") +
    geom_point(alpha=0.7, size=0.1) + 
    scale_colour_gradient(
  low = "#7AC6FF",
  high = "#023768",
  space = "Lab",
  na.value = "grey50",
  guide = "colourbar",
  aesthetics = "colour"
)+
    #scale_color_manual(values=c("Metabolomics"="black"))+
    scale_y_continuous(limits=c(0, NA), expand=expansion(mult=c(0, .05)))+#, limits=c(0, NA))+
    scale_x_continuous(expand=expansion(add=c(0, 1)))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = endpoint_map), ncol=25) + theme(legend.position="none")

In [ ]:
plot_name = "MedicalHistoryRisk"
mh_events %>% ggsave(filename=glue("outputs/{plot_name}.png"), device="png", width=plot_width, height=plot_height, dpi=plot_res, limitsize=FALSE)

## Add buffer

In [ ]:
pred_outcomes %>% head()

In [ ]:
logh_T_agg_buffer = pred_outcomes %>% filter(prevalent==0) %>% mutate(event_buffer = case_when((event!=0&time<1) ~ 0, TRUE ~ event)) %>% group_by(endpoint, model) %>% mutate(logh_perc = ntile(logh, 100)) %>% group_by(endpoint, model, logh_perc) %>% summarise(ratio = mean(event_buffer)) %>% as_tibble()

In [ ]:
logh_T_agg_buffer %>% write_feather(glue("{output_path}/logh_agg_1ybuffer_220224.feather"))

In [ ]:
plot_width = 50; plot_height=75; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
#temp_rank = event_rest %>% filter(features=="Metabolomics") %>% arrange(desc(MET10PercvsREST))
endpoint_order = (phecode_defs %>% mutate(phecode_rank = as.numeric(phecode)) %>% arrange(phecode_rank) %>% as_tibble())$endpoint
endpoint_selection = endpoint_order# %>% head(500)
temp = logh_T_agg_buffer %>% filter(model=="GNN(Records)+MLP") %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% filter(endpoint %in% endpoint_selection) %>% ungroup() #%>% sample_n(10000)

mh_events = ggplot(temp, aes(x=logh_perc, y=ratio*100, color=logh_perc)) + 
    labs(title=NULL, x="Medical History Percentile [%]", y="Observed Event Rate [%]") +
    geom_point(alpha=0.7, size=0.1) + 
    scale_colour_gradient(
  low = "#7AC6FF",
  high = "#023768",
  space = "Lab",
  na.value = "grey50",
  guide = "colourbar",
  aesthetics = "colour"
)+
    #scale_color_manual(values=c("Metabolomics"="black"))+
    scale_y_continuous(limits=c(0, NA), expand=expansion(mult=c(0, .05)))+#, limits=c(0, NA))+
    scale_x_continuous(expand=expansion(add=c(0, 1)))+
    facet_wrap(~endpoint, scale="free", labeller = labeller(endpoint = endpoint_map), ncol=25) + theme(legend.position="none")

In [ ]:
plot_name = "MedicalHistoryRisk_1ybuffer"
mh_events %>% ggsave(filename=glue("outputs/{plot_name}.png"), device="png", width=plot_width, height=plot_height, dpi=plot_res, limitsize=FALSE)

## Metabolomic State and Event Trajectories

In [ ]:
logh_mh = logh_inc %>% select(endpoint, model, eid, logh_perc, event, time) %>% group_by(endpoint) %>% 
    mutate(MH=case_when(logh_perc %in% 91:100 ~ "High", 
                        logh_perc %in% 45:55 ~ "Mid", 
                        logh_perc %in% 1:10 ~ "Low",
                        TRUE ~ "NA")
          ) %>% mutate(MET = fct_rev(factor(MET, levels=c("Low", "Mid", "High")))) %>% ungroup() %>% 
    filter(MH!="NA") #%>% select(eid, endpoint, logh, logh_group)

In [ ]:
require("ggquickeda")
plot_width = 50; plot_height=75; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
met_map = c("High"="#023768", "Mid"="#4F8EC1", "Low"="#7AC6FF")

endpoint_order = (phecode_defs %>% mutate(phecode_rank = as.numeric(phecode)) %>% arrange(phecode_rank) %>% as_tibble())$endpoint
endpoint_selection = endpoint_order# %>% head(500)
temp = logh_mh %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% filter(endpoint %in% endpoint_selection) %>% ungroup() 

km_plot = ggplot(temp, aes(time = time, status = event, fill=MET, color=MET,group=MET)) +
    geom_km(trans = "event") + 
    geom_kmticks(trans = "event", size=0.3) + geom_kmband(trans = "event") + 
    labs(x="Time [Years]", y="Cumulative Events [%]")+
    scale_color_manual(values=met_map)+scale_fill_manual(values=met_map)+
    scale_y_continuous(labels = function(x) round(x*100, 1), expand=c(0, 0))+
    scale_x_continuous(expand=expansion(add=c(0, .1)), breaks=c(5, 10))+
    facet_wrap(~ endpoint, scale="free", labeller = labeller(endpoint = endpoint_map), ncol=25) + theme(legend.position="none")

In [ ]:
plot_name = "MedicalHistory_KMs"
km_plot %>% ggsave(filename=glue("outputs/{plot_name}.png"), device="png", width=plot_width, height=plot_height, dpi=plot_res, limitsize=FALSE)

# Top 1%

In [ ]:
logh_mh = logh_inc %>% select(endpoint, model, eid, logh_perc, event, time) %>% group_by(endpoint) %>% 
    mutate(MH=case_when(logh_perc == 100 ~ "High", 
                        logh_perc %in% 50:51 ~ "Mid", 
                        logh_perc == 1 ~ "Low",
                        TRUE ~ "NA")
          ) %>% mutate(MET = fct_rev(factor(MH, levels=c("Low", "Mid", "High")))) %>% ungroup() %>% 
    filter(MH!="NA") #%>% select(eid, endpoint, logh, logh_group)

In [ ]:
require("ggquickeda")
plot_width = 50; plot_height=75; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
mh_map = c("High"="#023768", "Mid"="#4F8EC1", "Low"="#7AC6FF")

endpoint_order = (phecode_defs %>% mutate(phecode_rank = as.numeric(phecode)) %>% arrange(phecode_rank) %>% as_tibble())$endpoint
endpoint_selection = endpoint_order# %>% head(500)
temp = logh_mh %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% filter(endpoint %in% endpoint_selection) %>% ungroup() 

km_plot = ggplot(temp, aes(time = time, status = event, fill=MH, color=MH,group=MH)) +
    geom_km(trans = "event") + 
    geom_kmticks(trans = "event", size=0.3) + geom_kmband(trans = "event") + 
    labs(x="Time [Years]", y="Cumulative Events [%]")+
    scale_color_manual(values=met_map)+scale_fill_manual(values=mh_map)+
    scale_y_continuous(labels = function(x) round(x*100, 1), expand=c(0, 0))+
    scale_x_continuous(expand=expansion(add=c(0, .1)), breaks=c(5, 10))+
    facet_wrap(~ endpoint, scale="free", labeller = labeller(endpoint = endpoint_map), ncol=25) + theme(legend.position="none")

In [ ]:
plot_name = "MedicalHistory_KMs_Top1"
km_plot %>% ggsave(filename=glue("outputs/{plot_name}.png"), device="png", width=plot_width, height=plot_height, dpi=plot_res, limitsize=FALSE)

# Figure 2

In [ ]:
plot_width=8.25; plot_height=10; plot_dpi=320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_dpi)
fig2 = met_events / km_plot# +plot_annotation(tag_levels = 'A')

In [ ]:
fig2

In [ ]:
library(gt)
plot_name = "Figures_2_AB"
fig2 %>% ggsave(filename=glue("outputs/{plot_name}.pdf"), device="pdf", width=plot_width, height=plot_height, dpi=320)